# Robustness of Image Metrics for Magnetic Domain Images

**Purpose**: Demonstrate that pixel-based metrics (MSE, BCE, SSIM) are brittle under physically irrelevant transformations (small shifts, reflections) of magnetic domain images, while cosine similarity in the Xception latent space remains stable. All scores are reported **normalized** (SSIM/cosine relative to E0; MSE/BCE relative to E2).
**Inputs**: `dataset_unificado_v2.npz`, `modelo_xception_fulldatabaseV3100.h5`, shared `metrics.py` (Kaggle dataset `carloscanamejoy/physicalmetrics`).
**Outputs**: Figures in `OUTPUT_DIR/robustness/` (`robustness_metrics_summary`, `robustness_metrics_per_structure`, `robustness_qualitative`, `robustness_cosine_vs_ssim_scatter`, each as `.png` + `.svg`) and `robustness_scores.csv` (raw + normalized columns).
**Execution environment**: Kaggle / Google Colab (run only one setup cell).
**Dependencies**: tensorflow, scikit-learn, scikit-image, matplotlib, seaborn, scipy, tqdm, numpy, pandas

This is a standalone experiment using **only real images** — no generative model is needed.
It runs on the **complete test split** (~25,451 images; 70/15/15 stratified split, SEED=42).
Magnetic domain patterns are translation- and reflection-equivariant under the Hamiltonian:
a shifted or mirrored configuration is physically equivalent. A good similarity metric
should therefore be tolerant to these transformations; pixel-aligned metrics are not.

## Kaggle Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — KAGGLE
# Run this cell only when executing on Kaggle.
# All datasets (including metrics.py from the physicalmetrics
# dataset) are pre-mounted read-only under:
#   /kaggle/input/datasets/carloscanamejoy/
# Nothing to download — paths are defined in the Shared
# Configuration cell below.
# ============================================================
import os, sys
print("Kaggle environment ready:", os.path.exists("/kaggle/input"))

## Google Colab Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — GOOGLE COLAB
# Run this cell only when executing on Google Colab.
# Datasets are downloaded via the Kaggle API using kaggle.json.
# Upload your kaggle.json file to the Colab session before running.
# ============================================================
import os, sys, shutil, zipfile
from google.colab import files

# --- Kaggle credentials ---
os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()                          # upload kaggle.json when prompted
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# --- Install dependencies ---
os.system("pip install -q kaggle torch torchvision tensorflow scikit-learn scikit-image tqdm matplotlib seaborn")

# --- Download datasets ---
os.system("kaggle datasets download -d carloscanamejoy/dataset-spines-united-v2   -p /content/data    --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-xception-model      -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-models              -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-cvae-models         -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/physicalmetrics             -p /content/weights --unzip")

## Load Shared Metrics

In [ ]:
# ── Load shared metrics module from Kaggle dataset ───────────────────────────
# On Kaggle: metrics.py is pre-mounted as part of the physicalmetrics dataset.
# On Colab:  metrics.py is downloaded via the Kaggle API along with the other datasets.

import importlib.util, sys, os

_METRICS_KAGGLE = "/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
_METRICS_COLAB  = "/content/weights/metrics.py"
_METRICS_LOCAL  = os.path.join("..", "utils", "metrics.py")    # running from the repo
_metrics_path   = next(p for p in (_METRICS_KAGGLE, _METRICS_COLAB, _METRICS_LOCAL)
                       if os.path.exists(p))

spec    = importlib.util.spec_from_file_location("metrics", _metrics_path)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics

from metrics import (
    STRUCTURE_MAP, STRUCTURE_NAMES, STRUCTURE_COLORS, MODEL_COLORS, PARAM_NAMES,
    circular_mask, MASK,
    masked_mse, masked_bce, masked_ssim,
    cosine_similarity_pair, cosine_similarity_batch,
    magnetization, abs_magnetization, cnn_correlation,
    structure_factor, azimuthal_average, oz_fit, chi_ensemble,
    shift_image, reflect_image,
    normalize_metrics,
    save_figure, apply_figure_style,
    center_crop, get_structure_label,
)
print(f"metrics module loaded from: {_metrics_path}")

## Shared Configuration

Auto-detects the execution environment (`_ON_KAGGLE`) and defines every path used
by this notebook — the rest of the code never hardcodes paths.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

SEED = 42

# --- Environment auto-detection: single source of truth for all paths ---
_ON_KAGGLE = os.path.exists("/kaggle/input")

DATASET_PATH  = ("/kaggle/input/datasets/carloscanamejoy/dataset-spines-united-v2/dataset_unificado_v2.npz"
                 if _ON_KAGGLE else "/content/data/dataset_unificado_v2.npz")
XCEPTION_PATH = ("/kaggle/input/datasets/carloscanamejoy/weights-xception-model/modelo_xception_fulldatabaseV3100.h5"
                 if _ON_KAGGLE else "/content/weights/modelo_xception_fulldatabaseV3100.h5")
DDPM_PATH     = ("/kaggle/input/datasets/carloscanamejoy/weights-models/ddpm_spines_final_39crop.pt"
                 if _ON_KAGGLE else "/content/weights/ddpm_spines_final_39crop.pt")
CVAEXPN_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_xception.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_xception.h5")
CVAEVIT_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_vit.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_vit.h5")
METRICS_PATH  = ("/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
                 if _ON_KAGGLE else "/content/weights/metrics.py")
OUTPUT_DIR    = "/kaggle/working/outputs" if _ON_KAGGLE else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

apply_figure_style()
FIG_DIR = os.path.join(OUTPUT_DIR, "robustness")
os.makedirs(FIG_DIR, exist_ok=True)

# Conditions and their display palette
CONDITION_KEYS   = ["E0", "E1", "E2", "E3"]
CONDITION_LABELS = {"E0": "E0 (identity)", "E1": "E1 (+1 px shift)",
                    "E2": "E2 (+5 px shift)", "E3": "E3 (reflection)"}
CONDITION_COLORS = {"E0": "#4C72B0", "E1": "#DD8452", "E2": "#55A868", "E3": "#C44E52"}
METRIC_KEYS = ["mse", "bce", "ssim", "cosine"]

## Section 1 — Data Loading & Preprocessing

Load the dataset and rebuild the 70/15/15 stratified split (`SEED=42`).
The experiment runs on the **complete test split** — no further subsampling.
Images are already in the [-1, 1] range.

In [ ]:
data     = np.load(DATASET_PATH, mmap_mode="r")
print(f"npz keys: {data.files}")
imgs     = data["img"]       # (N, 39, 39, 1)
params   = data["params"]    # (N, 8)
clusters = np.asarray(data["labels"]).astype(int)
all_idx  = np.arange(len(imgs))

idx_train, idx_temp = train_test_split(all_idx, test_size=0.30, random_state=SEED,
                                       stratify=clusters)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED,
                                     stratify=clusters[idx_temp])

# COMPLETE test split — no further subsampling
idx_test = np.sort(idx_test)
ref_imgs = np.asarray(imgs[idx_test], dtype=np.float32)
if ref_imgs.ndim == 4:
    ref_imgs = ref_imgs[..., 0]                              # (N_test, 39, 39)
clusters_test = clusters[idx_test]
structure_labels = np.array([get_structure_label(c) for c in clusters_test])

print(f"Test split size: {len(idx_test)} samples")
print({s: int((structure_labels == s).sum()) for s in STRUCTURE_NAMES})

## Section 2 — Circular Disk Mask

All pixel-level metrics are computed inside the inscribed circular disk of the
39×39 image (the physically simulated region). Pixels outside the mask are zeroed.

In [ ]:
print(f"Circular mask: {MASK.shape}, {MASK.sum()} of {MASK.size} pixels inside the disk")
assert MASK.shape == (39, 39), "circular_mask must be h=39, w=39"

fig, ax = plt.subplots(figsize=(3.2, 3.2))
ax.imshow(MASK, cmap="gray")
ax.set_title("Circular disk mask (39×39)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Section 3 — Four Robustness Conditions

For each reference image we build four comparison pairs:
- **E0** — `(img, img)`: identity, the metric's ideal score.
- **E1** — `(img, shift +1 px)`: 1-pixel horizontal shift (`np.roll`).
- **E2** — `(img, shift +5 px)`: 5-pixel horizontal shift.
- **E3** — `(img, reflection)`: horizontal flip.

All transformed images describe the *same physical state* as the original.

In [ ]:
def transform_condition(stack, key):
    """Apply a robustness condition to a (B, 39, 39) image stack (vectorized)."""
    if key == "E0":
        return stack
    if key == "E1":
        return np.roll(stack, 1, axis=2)        # shift_image(img, 1, axis=1) per image
    if key == "E2":
        return np.roll(stack, 5, axis=2)        # shift_image(img, 5, axis=1) per image
    if key == "E3":
        return stack[:, :, ::-1]                # reflect_image(img) per image
    raise ValueError(key)


# Quick visual check on one example
example = ref_imgs[0]
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, key in zip(axes, CONDITION_KEYS):
    ax.imshow(transform_condition(example[None], key)[0], cmap="jet", vmin=-1, vmax=1)
    ax.set_title(CONDITION_LABELS[key], fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Section 4 — Metric Computation & Normalization

Pixel metrics (MSE, BCE, SSIM) are computed on masked images; cosine similarity
uses 256-dim features from the Xception `Dense(256)` layer (GAP → BN → Dropout →
Dense). This notebook has no PyTorch model competing for the GPU, so Xception
inference runs on GPU when available (falling back to CPU). Features for the
~25,451 × 5 image stacks are cached to `OUTPUT_DIR` so re-runs skip extraction.

Raw scores are then normalized with `normalize_metrics()`:
SSIM/cosine relative to the E0 mean (E0 = 1.0), MSE/BCE relative to the E2 mean
(E2 = 1.0, E0 ≈ 0.0). All figures use the normalized scores.

In [ ]:
import tensorflow as tf

# TensorFlow: allow GPU memory growth (prevents TF from claiming all VRAM)
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# No PyTorch model in this notebook → safe to use the GPU for Xception inference
XCEPTION_DEVICE = "/gpu:0" if tf.config.list_physical_devices("GPU") else "/cpu:0"
print(f"Xception inference device: {XCEPTION_DEVICE}")


def build_xception(n_out=8):
    inp  = tf.keras.Input(shape=(224, 224, 3))
    base = tf.keras.applications.Xception(weights=None, include_top=False, input_tensor=inp)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.BatchNormalization(name="batch_normalization_4")(x)
    x = tf.keras.layers.Dropout(0.4, name="dropout")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="dense")(x)
    x = tf.keras.layers.BatchNormalization(name="batch_normalization_5")(x)
    x = tf.keras.layers.Dropout(0.3, name="dropout_1")(x)
    return tf.keras.Model(inp, tf.keras.layers.Dense(n_out, activation="linear", name="dense_1")(x))


with tf.device(XCEPTION_DEVICE):
    xception_model = build_xception()
    xception_model.load_weights(XCEPTION_PATH)
print(f"Xception loaded — {xception_model.count_params():,} parameters")

feature_extractor = tf.keras.Model(
    inputs=xception_model.input,
    outputs=xception_model.get_layer("dense").output)


def extract_xception_features(imgs_batch, batch_size=64, chunk=2048):
    """
    Extract 256-dim features from the Dense(256) layer.
    imgs_batch: (B, 39, 39) numpy array, values in [-1, 1]. Returns (B, 256).
    Processes in chunks so the 224×224 RGB intermediate never exceeds ~2 GB.
    """
    out = np.empty((len(imgs_batch), 256), dtype=np.float32)
    for i0 in tqdm(range(0, len(imgs_batch), chunk), desc="features", leave=False):
        i1 = min(i0 + chunk, len(imgs_batch))
        x = tf.image.resize(imgs_batch[i0:i1, ..., np.newaxis], (224, 224)).numpy()
        x = np.repeat(x, 3, axis=-1)
        # Scale [-1,1] → [0,255] then apply Xception preprocessing (maps back to [-1,1])
        x = (x + 1) / 2.0
        x = tf.keras.applications.xception.preprocess_input(x * 255.0)
        with tf.device(XCEPTION_DEVICE):
            out[i0:i1] = feature_extractor.predict(x, batch_size=batch_size, verbose=0)
    return out

In [ ]:
# --- Raw scores over the complete test split, per condition ---
n_test = len(ref_imgs)
mask_flat = MASK.ravel()
ref_masked = ref_imgs * MASK                                 # zero outside the disk

raw_scores = {}
FEAT_CACHE = os.path.join(OUTPUT_DIR, f"robustness_features_N{n_test}.npz")
feats = dict(np.load(FEAT_CACHE)) if os.path.exists(FEAT_CACHE) else {}

for key in CONDITION_KEYS:                       # E0 runs first → feats["E0"] is z_ref
    var = transform_condition(ref_imgs, key)
    var_masked = var * MASK

    # Vectorized masked MSE / BCE over all images at once
    a = ref_masked.reshape(n_test, -1)[:, mask_flat]
    b = var_masked.reshape(n_test, -1)[:, mask_flat]
    mse = ((a - b) ** 2).mean(axis=1)
    a01, b01 = (a + 1) / 2.0, (b + 1) / 2.0
    eps = 1e-7
    bce = -(a01 * np.log(b01 + eps) + (1 - a01) * np.log(1 - b01 + eps)).mean(axis=1)

    # SSIM per image (skimage)
    ssim_arr = np.array([masked_ssim(ref_masked[i], var_masked[i])
                         for i in tqdm(range(n_test), desc=f"SSIM {key}", leave=False)])

    # Cosine similarity in Xception latent space (features cached per condition)
    if key not in feats:
        feats[key] = extract_xception_features(var)
        np.savez_compressed(FEAT_CACHE, **feats)
    cos = cosine_similarity_batch(feats["E0"], feats[key])

    raw_scores[key] = {"mse": mse, "bce": bce, "ssim": ssim_arr, "cosine": cos}
    print(f"{CONDITION_LABELS[key]}: raw means "
          f"mse={mse.mean():.4f} bce={bce.mean():.4f} "
          f"ssim={ssim_arr.mean():.4f} cosine={cos.mean():.4f}")

# --- Normalization (Correction 7 convention) ---
norm_scores, denominators = normalize_metrics(raw_scores, reference_key="E0", worst_key="E2")

print("\nNormalization denominators:")
for m, v in denominators.items():
    print(f"  {m}: {v:.6f}")

# Sanity check: E0 SSIM/Cosine should be 1.0; E0 MSE/BCE should be ~0.0
print("\nNormalized condition means:")
for cond in CONDITION_KEYS:
    means = {m: float(np.mean(norm_scores[cond][m])) for m in METRIC_KEYS}
    print(f"  {cond}: { {m: round(v, 4) for m, v in means.items()} }")

# --- Save raw + normalized scores to CSV ---
rows = []
for cond in CONDITION_KEYS:
    for i in range(len(idx_test)):
        rows.append({
            "condition":    cond,
            "structure":    structure_labels[i],
            "mse_raw":      raw_scores[cond]["mse"][i],
            "bce_raw":      raw_scores[cond]["bce"][i],
            "ssim_raw":     raw_scores[cond]["ssim"][i],
            "cosine_raw":   raw_scores[cond]["cosine"][i],
            "mse_norm":     norm_scores[cond]["mse"][i],
            "bce_norm":     norm_scores[cond]["bce"][i],
            "ssim_norm":    norm_scores[cond]["ssim"][i],
            "cosine_norm":  norm_scores[cond]["cosine"][i],
        })

pd.DataFrame(rows).to_csv(f"{OUTPUT_DIR}/robustness/robustness_scores.csv", index=False)
print(f"\nSaved {len(rows):,} rows to robustness_scores.csv (raw + normalized columns)")

## Section 5 — Figures

All figures use the **normalized** scores: SSIM/cosine relative to E0 (1.0 = best),
MSE/BCE relative to E2 (0.0 = best, 1.0 = 5-px-shift level).

**Figure 1** — overall normalized mean ± std per metric and condition.

In [ ]:
NORM_ANNOTATION = ("SSIM & Cosine normalized to E0 (identical images) = 1.0  |  "
                   "MSE & BCE normalized to E2 (5-px shift) = 1.0")
METRICS = [("mse", "Normalized MSE (relative to E2)", "Pixel-aligned: explodes under any shift"),
           ("bce", "Normalized BCE (relative to E2)", "Pixel-aligned: same failure mode as MSE"),
           ("ssim", "Normalized SSIM (relative to E0)", "Structural but still alignment-sensitive"),
           ("cosine", "Normalized Cosine Similarity (relative to E0)",
            "Latent-space: stable under shift/reflection")]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (key, title, note) in zip(axes.flat, METRICS):
    means = [np.mean(norm_scores[c][key]) for c in CONDITION_KEYS]
    stds  = [np.std(norm_scores[c][key]) for c in CONDITION_KEYS]
    ax.bar(range(4), means, yerr=stds, capsize=3,
           color=[CONDITION_COLORS[c] for c in CONDITION_KEYS])
    ax.axhline(means[0], ls="--", color="gray", lw=1, label="E0 reference")
    ax.set_xticks(range(4)); ax.set_xticklabels(CONDITION_KEYS)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"{title}\n{note}", fontsize=10)
    ax.set_xlabel("Condition"); ax.set_ylabel("Normalized score"); ax.legend()
fig.suptitle("Metric robustness under physically equivalent transformations "
             f"(complete test split, N = {n_test:,})")
fig.text(0.01, 0.01, NORM_ANNOTATION, fontsize=7, color="gray", ha="left", va="bottom")
save_figure(fig, os.path.join(FIG_DIR, "robustness_metrics_summary"))
plt.show()

**Figure 2** — per-structure breakdown: each normalized metric split by the six magnetic structures.

In [ ]:
n_struct = len(STRUCTURE_NAMES)
bar_w = 0.8 / n_struct

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (key, title, _) in zip(axes.flat, METRICS):
    for si, s in enumerate(STRUCTURE_NAMES):
        sm = structure_labels == s
        means = [np.mean(norm_scores[c][key][sm]) for c in CONDITION_KEYS]
        stds  = [np.std(norm_scores[c][key][sm]) for c in CONDITION_KEYS]
        ax.bar(np.arange(4) + (si - (n_struct - 1) / 2) * bar_w, means, bar_w,
               yerr=stds, capsize=1.5, color=STRUCTURE_COLORS[s], label=s)
    ax.set_xticks(range(4)); ax.set_xticklabels(CONDITION_KEYS)
    ax.set_ylim(0, 1.1)
    ax.set_title(title); ax.set_xlabel("Condition"); ax.set_ylabel("Normalized score")
axes[0, 0].legend(fontsize=7, ncol=2)
fig.suptitle("Metric robustness per magnetic structure (normalized scores)")
fig.text(0.01, 0.01, NORM_ANNOTATION, fontsize=7, color="gray", ha="left", va="bottom")
save_figure(fig, os.path.join(FIG_DIR, "robustness_metrics_per_structure"))
plt.show()

**Figure 3** — qualitative grid: 2 examples per structure (12 rows), original vs the
four conditions, with normalized MSE / SSIM / cosine values printed below each
transformed image.

In [ ]:
rng = np.random.RandomState(SEED + 1)
example_rows = []
for s in STRUCTURE_NAMES:
    pool = np.where(structure_labels == s)[0]
    example_rows.extend(rng.choice(pool, min(2, len(pool)), replace=False))

col_titles = ["Original", "E0 (ref)", "E1 (+1 px)", "E2 (+5 px)", "E3 (reflected)"]
n_rows = len(example_rows)
fig, axes = plt.subplots(n_rows, 5, figsize=(11, 2.4 * n_rows))
for r, i in enumerate(example_rows):
    axes[r, 0].imshow(ref_imgs[i], cmap="jet", vmin=-1, vmax=1)
    axes[r, 0].set_ylabel(structure_labels[i].replace(" & ", "\n& "), fontsize=7)
    for ci, key in enumerate(CONDITION_KEYS):
        ax = axes[r, ci + 1]
        ax.imshow(transform_condition(ref_imgs[i][None], key)[0],
                  cmap="jet", vmin=-1, vmax=1)
        ax.set_xlabel(f"MSEn={norm_scores[key]['mse'][i]:.3f}\n"
                      f"SSIMn={norm_scores[key]['ssim'][i]:.3f}\n"
                      f"Cosn={norm_scores[key]['cosine'][i]:.3f}", fontsize=7)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
for j, t in enumerate(col_titles):
    axes[0, j].set_title(t)
fig.suptitle("Qualitative robustness: identical physics, divergent pixel metrics "
             "(normalized scores)", y=1.0)
fig.text(0.01, 0.005, NORM_ANNOTATION, fontsize=7, color="gray", ha="left", va="bottom")
save_figure(fig, os.path.join(FIG_DIR, "robustness_qualitative"))
plt.show()

**Figure 4** — SSIM vs cosine scatter: cosine similarity stays near 1 in the regions
where SSIM collapses, which is the core argument for latent-space evaluation.

In [ ]:
struct_markers = {"Ferromagnetic": "o", "Helical": "D", "Labyrinthine & Conical": "^",
                  "Bimeron": "s", "Skyrmions": "P", "Field-Saturated": "X"}
fig, ax = plt.subplots(figsize=(7.5, 6))
for c in CONDITION_KEYS:
    for s in STRUCTURE_NAMES:
        sm = structure_labels == s
        ax.scatter(norm_scores[c]["ssim"][sm], norm_scores[c]["cosine"][sm],
                   s=6, alpha=0.25, color=CONDITION_COLORS[c],
                   marker=struct_markers[s], rasterized=True,
                   label=CONDITION_LABELS[c] if s == STRUCTURE_NAMES[0] else None)
ax.set_xlabel("Normalized SSIM (relative to E0)")
ax.set_ylabel("Normalized Cosine Similarity (relative to E0)")
ax.set_title("Cosine similarity is stable where SSIM degrades\n"
             "(colors = condition, marker shape = structure)")
leg = ax.legend(title="Condition", fontsize=8)
for handle in leg.legend_handles:
    handle.set_alpha(1.0)
fig.text(0.01, 0.01, NORM_ANNOTATION, fontsize=7, color="gray", ha="left", va="bottom")
save_figure(fig, os.path.join(FIG_DIR, "robustness_cosine_vs_ssim_scatter"))
plt.show()